# [미션] 데이터 전처리 복습용 문제



---



## **Dataset**: Kaggle 데이터셋 버전

| 문제 | Kaggle 데이터셋 | 배우는 것 | 난이도 |
|---|---|---|---|
| ① 신입사원(취업) 합격 예측 | `benroshan/factors-affecting-campus-placement` — Campus Recruitment (215명) | X / y 구분, Encoding · Scaling · Imputation, **미래 정보(salary) 누수** | [입문] |
| ② 영화 평점 | `shubhammehta21/movie-lens-small-latest-dataset` — MovieLens small (10만 평점) | **평점 없음 ≠ 평점 0**, 결측치 처리의 사고 | [중급] |
| ③ 시험문제 유출 | `uciml/pima-indians-diabetes-database` — Pima Diabetes (768명) | **Data Leakage**: 분리 → Train fit → Test transform, Pipeline | [도전★] |

> **실행 환경**
> * **Kaggle Notebook**: 오른쪽 *Add Input* 에서 위 3개 데이터셋을 추가하면 `/kaggle/input/...` 에서 자동으로 읽습니다.
> * **Colab / 로컬**: `pip install kagglehub` 후 최초 1회 Kaggle 로그인(API 토큰)이 필요합니다. `kagglehub.dataset_download()` 가 내려받습니다.
> * 네트워크가 없으면 `DEMO_MODE = True` — 같은 컬럼 구조의 **모의 데이터**로 코드 흐름만 확인합니다 (수치는 교육적 의미 없음).
>

## 0. 준비 — 공통 로더

In [ ]:
import os, glob, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

DEMO_MODE = False          # Kaggle 접근이 안 되면 True 로 바꾸세요 (모의 데이터로 흐름만 확인)

def load_kaggle(handle: str, filename: str) -> pd.DataFrame:
    """Kaggle 데이터셋의 CSV 한 개를 DataFrame 으로.
    우선순위: ① Kaggle Notebook 의 /kaggle/input  ② kagglehub 다운로드  ③ DEMO_MODE 모의 데이터"""
    if not DEMO_MODE:
        # ① Kaggle Notebook 환경
        hits = glob.glob(f"/kaggle/input/**/{filename}", recursive=True)
        if hits:
            print(f"[kaggle input] {hits[0]}"); return pd.read_csv(hits[0])
        # ② Colab / 로컬 — kagglehub
        try:
            import kagglehub
            path = kagglehub.dataset_download(handle)
            hits = glob.glob(f"{path}/**/{filename}", recursive=True)
            print(f"[kagglehub] {hits[0]}"); return pd.read_csv(hits[0])
        except Exception as e:
            print(f"[kagglehub 실패 → DEMO 데이터 사용] {type(e).__name__}: {str(e)[:80]}")
    return make_demo(filename)

def make_demo(filename: str) -> pd.DataFrame:
    """실제 Kaggle 파일과 같은 컬럼 구조의 모의 데이터 (교육용 흐름 확인 전용)"""
    rng = np.random.default_rng(42)
    if filename == "Placement_Data_Full_Class.csv":
        n = 215
        df = pd.DataFrame({
            "sl_no": np.arange(1, n + 1),
            "gender": rng.choice(["M", "F"], n),
            "ssc_p": rng.normal(67, 10, n).round(2), "ssc_b": rng.choice(["Central", "Others"], n),
            "hsc_p": rng.normal(66, 11, n).round(2), "hsc_b": rng.choice(["Central", "Others"], n),
            "hsc_s": rng.choice(["Commerce", "Science", "Arts"], n, p=[.5, .4, .1]),
            "degree_p": rng.normal(66, 7, n).round(2),
            "degree_t": rng.choice(["Comm&Mgmt", "Sci&Tech", "Others"], n, p=[.67, .27, .06]),
            "workex": rng.choice(["No", "Yes"], n, p=[.65, .35]),
            "etest_p": rng.uniform(50, 98, n).round(2),
            "specialisation": rng.choice(["Mkt&Fin", "Mkt&HR"], n, p=[.56, .44]),
            "mba_p": rng.normal(62, 6, n).round(2),
        })
        score = (df.ssc_p + df.hsc_p + df.degree_p) / 3 + 5 * (df.workex == "Yes") + rng.normal(0, 6, n)
        df["status"] = np.where(score > 66, "Placed", "Not Placed")
        df["salary"] = np.where(df.status == "Placed", rng.normal(288000, 90000, n).round(-3), np.nan)
        return df
    if filename == "movies.csv":
        n = 300
        return pd.DataFrame({"movieId": np.arange(1, n + 1),
                             "title": [f"Demo Movie {i} ({1990 + i % 30})" for i in range(1, n + 1)],
                             "genres": rng.choice(["Drama", "Comedy|Romance", "Action|Thriller", "Documentary"], n)})
    if filename == "ratings.csv":
        rows = []
        for u in range(1, 61):
            for m in rng.choice(np.arange(1, 281), rng.integers(15, 60), replace=False):   # 281~300 은 평점 없음
                rows.append((u, m, float(rng.choice([0.5, 1, 1.5, 2, 2.5, 3, 3.5, 4, 4.5, 5], p=[.02, .03, .03, .07, .08, .2, .15, .22, .1, .1])), 1_000_000_000 + u))
        return pd.DataFrame(rows, columns=["userId", "movieId", "rating", "timestamp"])
    if filename == "diabetes.csv":
        n = 768
        df = pd.DataFrame({
            "Pregnancies": rng.integers(0, 15, n), "Glucose": rng.normal(120, 30, n).round(),
            "BloodPressure": rng.normal(70, 12, n).round(), "SkinThickness": rng.normal(25, 10, n).clip(5).round(),
            "Insulin": rng.normal(100, 80, n).clip(15).round(), "BMI": rng.normal(32, 7, n).round(1),
            "DiabetesPedigreeFunction": rng.uniform(0.08, 2.4, n).round(3), "Age": rng.integers(21, 80, n)})
        for c, p in [("Glucose", .01), ("BloodPressure", .05), ("SkinThickness", .3), ("Insulin", .49), ("BMI", .015)]:
            df.loc[rng.random(n) < p, c] = 0          # 실제 데이터처럼 '0 = 측정 없음'
        z = 0.03 * (df.Glucose - 120) + 0.08 * (df.BMI - 32) + 0.02 * (df.Age - 40) + rng.normal(0, 1, n)
        df["Outcome"] = (z > 0.3).astype(int)
        return df
    raise ValueError(filename)

print("pandas", pd.__version__)

---
## **[문제] ① 신입사원(취업) 합격 예측** — Campus Recruitment

**Kaggle**: [benroshan/factors-affecting-campus-placement](https://www.kaggle.com/datasets/benroshan/factors-affecting-campus-placement) · 인도 경영대학원 졸업생 215명의 성적·전공·경력·취업 결과

| 슬라이드의 항목 | 데이터셋 컬럼 | 전처리 |
|---|---|---|
| 학점 | `ssc_p`, `hsc_p`, `degree_p`, `mba_p` (백분율 점수) | Scaling |
| 토익(시험 점수) | `etest_p` (취업 적성시험) | Scaling |
| 프로젝트 횟수 → 경력 | `workex` (Yes/No) | Encoding |
| 전공 | `degree_t`, `specialisation`, `hsc_s` | One-Hot Encoding |
| 합격 여부 | `status` (Placed / Not Placed) | **y** |
| (함정) | `salary` — 합격자만 값이 있고 불합격자는 NaN | **미래 정보 → 제거** |

**Q1.** 무엇이 X이고 무엇이 y일까?  **Q2.** `salary` 를 X에 넣으면 안 되는 이유는?

In [ ]:
df1 = load_kaggle("benroshan/factors-affecting-campus-placement", "Placement_Data_Full_Class.csv")
print(df1.shape)
display(df1.head())
print("\n자료형"); print(df1.dtypes)
print("\n결측치"); print(df1.isna().sum()[df1.isna().sum() > 0])
print("\nstatus 분포"); print(df1["status"].value_counts())

### 1-1. 결측치 `salary` 는 왜 비어 있나? — '모른다' 가 아니라 '해당 없음'
`salary` 가 NaN 인 행은 전부 `Not Placed` 다. 즉 결측 자체가 정답을 알려준다. 이런 열은 채우는 것이 아니라 **X에서 빼야** 한다 (예측 시점에는 알 수 없는 *미래 정보*).

In [ ]:
print(pd.crosstab(df1["status"], df1["salary"].isna(), rownames=["status"], colnames=["salary 결측?"]))

# 함정 확인: salary 결측 여부만으로 '예측' 하면?
leak_pred = np.where(df1["salary"].isna(), "Not Placed", "Placed")
print("\nsalary 결측 여부만으로 맞춘 비율 =", (leak_pred == df1["status"]).mean().round(3), "← 100%: 정답을 베낀 것")

### 1-2. X / y 정의 → 열 종류별 전처리 → Pipeline 학습

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

# X: 예측 시점에 알 수 있는 정보만.  y: status
X1 = df1.drop(columns=["sl_no", "status", "salary"])          # sl_no 는 일련번호(정보 없음), salary 는 미래 정보
y1 = (df1["status"] == "Placed").astype(int)

num_cols = ["ssc_p", "hsc_p", "degree_p", "etest_p", "mba_p"]                       # 학점·시험 점수 → Scaling
cat_cols = ["gender", "ssc_b", "hsc_b", "hsc_s", "degree_t", "workex", "specialisation"]   # 전공·경력 → Encoding

X1_tr, X1_te, y1_tr, y1_te = train_test_split(X1, y1, test_size=0.25, random_state=42, stratify=y1)

pre1 = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num_cols),
    ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                      ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))]), cat_cols),
])
model1 = Pipeline([("pre", pre1), ("clf", LogisticRegression(max_iter=1000))]).fit(X1_tr, y1_tr)
print(f"Test 정확도 = {model1.score(X1_te, y1_te):.3f}   (Train {len(X1_tr)}명 / Test {len(X1_te)}명)")

# 전처리 후 모델이 실제로 보는 열 이름
print("\n모델이 보는 특징 수:", len(model1.named_steps["pre"].get_feature_names_out()))
print(model1.named_steps["pre"].get_feature_names_out()[:12], "...")

### 1-3. 확인 질문
1. `etest_p` 가 비어 있는 학생이 새로 들어오면 이 파이프라인은 어떻게 처리하는가? (→ Train 의 중앙값)
2. `salary` 를 X에 다시 넣고 정확도를 재보라. 왜 1.0 이 되는가? 실무에서 이런 열을 찾아내는 방법은?
3. `hsc_s`(고교 계열)를 Label Encoding(Arts=0, Commerce=1, Science=2) 하면 무엇이 문제인가?

---
## **[문제] ② 영화 평점**  — MovieLens (small)

**Kaggle**: [shubhammehta21/movie-lens-small-latest-dataset](https://www.kaggle.com/datasets/shubhammehta21/movie-lens-small-latest-dataset) · 610명 사용자 × 9,742편 영화, 평점 100,836건 (GroupLens 배포판)

> 영화 A: ★★★★★ 영화 B: ★ 영화 C: **평가 없음** — 영화 C의 평점을 0점으로 처리해도 될까?

In [ ]:
movies  = load_kaggle("shubhammehta21/movie-lens-small-latest-dataset", "movies.csv")
ratings = load_kaggle("shubhammehta21/movie-lens-small-latest-dataset", "ratings.csv")
print(movies.shape, ratings.shape)
display(ratings.head(3))

# 영화별 평균 평점 · 평점 수
agg = ratings.groupby("movieId")["rating"].agg(mean="mean", count="count")
mv = movies.merge(agg, on="movieId", how="left")          # how="left": 평점이 하나도 없는 영화도 남긴다
no_rating = mv["count"].isna().sum()
print(f"\n전체 영화 {len(mv):,}편 중 평점이 하나도 없는 영화 = {no_rating}편  (mean 이 NaN)")
display(mv[mv["count"].isna()].head())

### 2-1. `fillna(0)` 을 하면 무슨 일이 생기나

In [ ]:
wrong = mv.copy(); wrong["mean"] = wrong["mean"].fillna(0)          # ✗ 평점 없음 → 0점
right = mv.copy()                                                     # ✓ NaN 그대로 (또는 별도 표시)

print("전체 영화 평균 평점  — fillna(0):", round(wrong["mean"].mean(), 3), "  NaN 유지:", round(right["mean"].mean(), 3))
print("\n'최악의 영화 10편' 을 뽑으면?  (fillna(0) 버전)")
display(wrong.sort_values("mean").head(10)[["title", "mean", "count"]])
print("→ 전부 '평가 없음' 영화. 0점은 '최악' 이 아니라 '모름' 이다.")

### 2-2. 올바른 처리 — 결측은 결측으로, 그리고 '몇 명이 평가했는가' 도 함께
평균만 보면 1명이 5점 준 영화가 1위가 된다. 실무(IMDb 등)에서는 **최소 평가 수**를 두거나, 전체 평균으로 끌어당기는 **가중 평균(베이지안 평균)** 을 쓴다.

In [ ]:
m = 20                                     # 최소 평가 수 (사람이 정하는 기준)
C = ratings["rating"].mean()               # 전체 평균 (사전 믿음)
mv["weighted"] = (mv["count"] * mv["mean"] + m * C) / (mv["count"] + m)   # 평가가 적으면 전체 평균 쪽으로

print("단순 평균 Top5 (1명이 5점 준 영화가 올라온다):")
display(mv.dropna(subset=["mean"]).sort_values("mean", ascending=False).head(5)[["title", "mean", "count"]])
print("가중 평균 Top5 (평가 수를 반영):")
display(mv.dropna(subset=["mean"]).sort_values("weighted", ascending=False).head(5)[["title", "mean", "count", "weighted"]])

# 평점이 없는 영화는 어떻게? — 세 가지 선택지
print("\n평점 없는 영화 처리 선택지:")
print("  ① 제외 (dropna)               → 추천 후보에서 빼기")
print("  ② 별도 플래그 열 has_rating=0  → 모델이 '정보 없음' 을 학습")
print("  ③ 전체 평균 C 로 채우기        → 0 이 아니라 '평균적인 영화' 로 가정")
mv["has_rating"] = mv["count"].notna().astype(int)
mv["mean_filled"] = mv["mean"].fillna(C)
display(mv[mv["has_rating"] == 0].head(3)[["title", "mean", "has_rating", "mean_filled"]])

### 2-3. 확인 질문
1. 사용자×영화 행렬(`ratings.pivot`)을 만들면 몇 %가 비어 있나? 그 빈칸을 0으로 채우면 추천 모델은 무엇을 배우게 되나?
2. 문제 ①의 `salary` 결측과 여기의 평점 결측은 성격이 어떻게 다른가? (해당 없음 vs 관측 안 됨)
3. `m` 을 5, 50, 200 으로 바꾸면 Top5 가 어떻게 변하나?

---
## **[문제] ③ 시험문제 유출 — Data Leakage** — Pima Indians Diabetes

**Kaggle**: [uciml/pima-indians-diabetes-database](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database) · 768명, 8개 임상 측정값 → 당뇨 여부 `Outcome`

> 100명의 데이터로 모델을 만들었다. **전체 100명의 평균·표준편차로 Scaling한 뒤** 80명 Train / 20명 Test 로 나눴다. 무엇이 잘못되었을까?

이 데이터에는 함정이 하나 더 있다 — `Glucose`, `BloodPressure`, `BMI`, `Insulin`, `SkinThickness` 의 **0 은 측정값이 아니라 '측정 없음'** 이다 (문제 ②와 같은 사고).

In [ ]:
df3 = load_kaggle("uciml/pima-indians-diabetes-database", "diabetes.csv")
print(df3.shape); display(df3.describe().round(1))

zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("\n0 인 값의 개수 (생리적으로 0 은 불가능 → 결측):")
print((df3[zero_cols] == 0).sum())
df3[zero_cols] = df3[zero_cols].replace(0, np.nan)      # 0 → NaN : '없음' 을 '없음' 으로
print("\n결측치로 바꾼 뒤:"); print(df3.isna().sum()[df3.isna().sum() > 0])

### 3-1. ✗ 잘못된 순서 — 전체 데이터로 Imputer·Scaler 를 fit 한 뒤 분리

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

X3 = df3.drop(columns=["Outcome"]); y3 = df3["Outcome"]

# ✗ 전체 X3 로 중앙값·평균·표준편차를 계산 → Test 정보가 전처리에 섞인다
imp_all = SimpleImputer(strategy="median").fit(X3)
sc_all  = StandardScaler().fit(imp_all.transform(X3))
X3_all  = sc_all.transform(imp_all.transform(X3))

X3_tr_w, X3_te_w, y3_tr, y3_te = train_test_split(X3_all, y3, test_size=0.2, random_state=42, stratify=y3)
knn_wrong = KNeighborsClassifier(n_neighbors=7).fit(X3_tr_w, y3_tr)
acc_wrong = knn_wrong.score(X3_te_w, y3_te)
print(f"✗ 누수 버전  Test 정확도 = {acc_wrong:.3f}")
print("   Scaler 가 본 Glucose 평균 =", round(sc_all.mean_[1], 2), "← Test 20% 의 값까지 포함해 계산됨")

### 3-2. ✓ 올바른 순서 — 먼저 분리, Train 에만 fit, Test 는 transform 만 (Pipeline 으로 강제)

In [ ]:
X3_tr, X3_te, y3_tr, y3_te = train_test_split(X3, y3, test_size=0.2, random_state=42, stratify=y3)

model3 = Pipeline([("imp", SimpleImputer(strategy="median")),
                   ("sc",  StandardScaler()),
                   ("knn", KNeighborsClassifier(n_neighbors=7))]).fit(X3_tr, y3_tr)   # fit 은 Train 에만
acc_right = model3.score(X3_te, y3_te)                                                # Test 는 transform 만
print(f"✓ 올바른 버전 Test 정확도 = {acc_right:.3f}")
print("   Scaler 가 본 Glucose 평균 =", round(model3.named_steps["sc"].mean_[1], 2), "← Train 80% 만으로 계산")
print(f"\n두 정확도 차이 = {abs(acc_wrong - acc_right):.3f}")

### 3-3. 차이가 작아 보이는데 왜 중요한가? — 교차검증으로 보면 누수는 **평가를 낙관적으로** 만든다
스케일링 누수는 이 데이터에서는 몇 % 안 된다. 그러나 (a) 누수는 **대체로 실제보다 좋아 보이는 방향**으로 작동하고(작을 때는 우연에 묻힐 수도 있다), (b) 더 강한 누수(타깃 인코딩, 중복 행, 미래 정보)는 수십 %를 만든다. 아래는 **의도적으로 강한 누수**를 넣어 크기를 비교한다.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# (A) 올바른 파이프라인 — 폴드마다 Train 부분에만 fit
acc_A = cross_val_score(Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                                  ("knn", KNeighborsClassifier(7))]), X3, y3, cv=cv).mean()

# (B) 전처리 누수 — 전체로 미리 변환한 뒤 CV
acc_B = cross_val_score(KNeighborsClassifier(7), X3_all, y3, cv=cv).mean()

# (C) 강한 누수 — 정답으로 만든 특징을 섞는다 (예: '진단 후 처방 여부' 같은 미래 정보를 흉내)
X3_leak = X3_all.copy()
X3_leak = np.c_[X3_leak, y3.values + np.random.default_rng(0).normal(0, 0.3, len(y3))]
acc_C = cross_val_score(KNeighborsClassifier(7), X3_leak, y3, cv=cv).mean()

# (D) 중복 행 누수 — 같은 사람이 Train 과 Test 에 모두 들어감
X3_dup = np.r_[X3_all, X3_all[:300]]; y3_dup = np.r_[y3.values, y3.values[:300]]
acc_D = cross_val_score(KNeighborsClassifier(7), X3_dup, y3_dup, cv=cv).mean()

print(pd.DataFrame({
    "상황": ["(A) 올바른 Pipeline", "(B) 전처리 누수 (전체로 Scaler fit)", "(C) 미래 정보 특징 누수", "(D) 중복 행 누수"],
    "5-fold 정확도": [round(a, 3) for a in [acc_A, acc_B, acc_C, acc_D]],
}).to_string(index=False))
print("\n→ (B) 스케일링 누수는 이 데이터에선 오차 범위 안이라 우연에 묻힐 수 있다. 그러나 (C)(D)처럼 누수가 커지면 시험 성적이 실제보다 확실히 좋아 보인다 — 크기가 작다고 습관을 허용하면 큰 누수도 놓친다.")